# LeetCode 1752 — Check if Array Is Sorted and Rotated bằng Python

Link đề: https://leetcode.com/problems/check-if-array-is-sorted-and-rotated/

Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def check(self, nums: list[int]) -> bool:

```

Mục tiêu học:

1. Hiểu phép **xoay mảng** (rotation) và công thức chỉ số vòng `B[i] = A[(i + x) % n]`.
2. Biết kiểm tra một mảng **không giảm** (non-decreasing) và dùng `all(...)`, `zip`, slicing.
3. Viết baseline **thử mọi phép xoay** để hiểu đề, rồi tìm ra **dấu hiệu đặc trưng**: mảng xoay từ mảng tăng có **tối đa một "điểm gãy"** khi đi vòng tròn.
4. Nắm vững **phép duyệt vòng** bằng `% n` — không quên cặp (phần tử cuối, phần tử đầu).
5. Xử lý trường hợp **phần tử trùng** (đề cho phép).
6. Tự giải biến thể: tìm số bước xoay, tìm phần tử nhỏ nhất, mảng không tăng xoay…

## 1. Bài toán yêu cầu gì?

Cho mảng `nums`. Trả về `True` nếu `nums` có thể thu được bằng cách lấy một mảng **sắp xếp không giảm** rồi **xoay** nó một số vị trí (kể cả 0 vị trí). Ngược lại trả `False`.

Mảng có thể có phần tử trùng.

Định nghĩa xoay: mảng `A` xoay `x` vị trí thành `B` cùng độ dài với `B[i] == A[(i + x) % A.length]`.

| nums | Kết quả | Vì sao? |
|---|---|---|
| `[3,4,5,1,2]` | `True` | xoay `[1,2,3,4,5]` 2 vị trí |
| `[2,1,3,4]` | `False` | không mảng tăng nào xoay ra được |
| `[1,2,3]` | `True` | xoay 0 vị trí |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

### Hình dung "xoay" như cắt và nối

Xoay `[1,2,3,4,5]` đi `x = 2` vị trí: lấy phần từ chỉ số 2 trở đi, nối với phần đầu.

```text
A           = [1, 2 | 3, 4, 5]
B = A[2:] + A[:2] = [3, 4, 5, 1, 2]
```

In [1]:
A = [1, 2, 3, 4, 5]
for x in range(len(A)):
    B = [A[(i + x) % len(A)] for i in range(len(A))]
    print(f"x={x}: {B}")

x=0: [1, 2, 3, 4, 5]
x=1: [2, 3, 4, 5, 1]
x=2: [3, 4, 5, 1, 2]
x=3: [4, 5, 1, 2, 3]
x=4: [5, 1, 2, 3, 4]


## 2. Nền tảng Python cần biết

### 2.1. Phép `%` (chia lấy dư) để đi vòng

`(i + x) % n` luôn nằm trong `0..n-1`. Khi `i + x` vượt quá cuối mảng, nó "quay lại" đầu. Ví dụ `n = 5`: `(4 + 2) % 5 = 1`.

In [2]:
n = 5
for k in range(8):
    print(k, "% 5 =", k % n)

0 % 5 = 0
1 % 5 = 1
2 % 5 = 2
3 % 5 = 3
4 % 5 = 4
5 % 5 = 0
6 % 5 = 1
7 % 5 = 2


### 2.2. Slicing `A[x:]` và `A[:x]`

- `A[x:]`: từ chỉ số `x` đến hết.
- `A[:x]`: từ đầu đến **trước** `x`.
- `A[x:] + A[:x]` chính là xoay trái `x` vị trí.

In [3]:
A = [1, 2, 3, 4, 5]
x = 2
print(A[x:], A[:x], A[x:] + A[:x])

[3, 4, 5] [1, 2] [3, 4, 5, 1, 2]


### 2.3. Kiểm tra mảng không giảm

"Không giảm" (non-decreasing): mỗi phần tử ≤ phần tử kế tiếp. Cho phép bằng nhau: `[1, 2, 2, 3]` là không giảm.

`all(điều_kiện for ...)` trả `True` nếu mọi điều kiện đúng (và `True` với dãy rỗng).

In [4]:
def is_non_decreasing(a):
    return all(a[i] <= a[i + 1] for i in range(len(a) - 1))

print(is_non_decreasing([1, 2, 2, 3]), is_non_decreasing([3, 1]), is_non_decreasing([7]))

True False True


`range(len(a) - 1)` vì ta đọc `a[i + 1]`; nếu `i` chạy tới `len(a) - 1` thì `a[i + 1]` vượt biên.

### 2.4. `zip(a, a[1:])` — cặp phần tử kề nhau

Cách viết "Pythonic" để duyệt các cặp `(a[i], a[i+1])`.

In [5]:
a = [3, 4, 5, 1, 2]
print(list(zip(a, a[1:])))

[(3, 4), (4, 5), (5, 1), (1, 2)]


### 2.5. `sorted(a)` trả list mới đã sắp xếp

Không thay đổi `a` gốc (khác với `a.sort()` sắp xếp tại chỗ).

In [6]:
a = [3, 4, 5, 1, 2]
print(sorted(a), a)

[1, 2, 3, 4, 5] [3, 4, 5, 1, 2]


### 2.6. `any(...)` — "có ít nhất một"

Đối xứng với `all`: `any` trả `True` nếu ít nhất một điều kiện đúng.

In [7]:
print(any([False, True, False]), any([False, False]))

True False


## 3. Cách trực giác — thử mọi phép xoay

Gợi ý của đề: "brute force, thử xem mảng sắp xếp có thể bắt đầu ở mỗi vị trí không".

Ý tưởng: `nums` là mảng xoay của một mảng không giảm ⇔ **xoay ngược** `nums` một số vị trí nào đó sẽ ra mảng không giảm. Ta thử mọi `x` từ `0` đến `n - 1`.

In [8]:
def check_bruteforce(nums: list[int]) -> bool:
    n = len(nums)
    for x in range(n):
        rotated = nums[x:] + nums[:x]     # thử coi nums[x] là phần tử đầu của mảng gốc
        if is_non_decreasing(rotated):
            return True
    return False

print(check_bruteforce([3, 4, 5, 1, 2]))   # True
print(check_bruteforce([2, 1, 3, 4]))      # False
print(check_bruteforce([1, 2, 3]))         # True

True
False
True


In [9]:
# Xem baseline thử những phép xoay nào cho [2, 1, 3, 4]
nums = [2, 1, 3, 4]
for x in range(len(nums)):
    r = nums[x:] + nums[:x]
    print(f"x={x}: {r} -> không giảm? {is_non_decreasing(r)}")

x=0: [2, 1, 3, 4] -> không giảm? False
x=1: [1, 3, 4, 2] -> không giảm? False
x=2: [3, 4, 2, 1] -> không giảm? False
x=3: [4, 2, 1, 3] -> không giảm? False


### 3.1. Một baseline khác: so với các phép xoay của `sorted(nums)`

Nếu `nums` là mảng xoay của mảng tăng, mảng tăng đó **phải** là `sorted(nums)` (cùng các phần tử). Ta thử mọi phép xoay của `sorted(nums)` xem có bằng `nums` không.

In [10]:
def check_compare_sorted(nums: list[int]) -> bool:
    s = sorted(nums)
    n = len(nums)
    return any(s[x:] + s[:x] == nums for x in range(n))

print(check_compare_sorted([3, 4, 5, 1, 2]), check_compare_sorted([2, 1, 3, 4]), check_compare_sorted([1, 2, 3]))

True False True


## 4. Nút thắt và insight

**Nút thắt:** mỗi lần thử tạo mảng mới `O(n)` và kiểm tra `O(n)` → `n` lần thử = `O(n²)`. Ta kiểm tra đi kiểm tra lại những cặp kề nhau giống hệt nhau.

**Quan sát:** đặt `nums` lên **vòng tròn** (phần tử cuối nối với phần tử đầu). Gọi một **điểm gãy** (drop) là vị trí `i` mà `nums[i] > nums[(i + 1) % n]`.

- Mảng tăng `[1,2,3,4,5]` trên vòng tròn: các cặp `1≤2, 2≤3, 3≤4, 4≤5` và cặp vòng `5 > 1` → **đúng 1 điểm gãy** (ở chỗ "nối đuôi về đầu").
- Xoay mảng chỉ **dời chỗ bắt đầu đọc** trên vòng tròn, **không đổi** các cặp kề nhau → mảng xoay vẫn có đúng 1 điểm gãy.
- Mảng mà mọi phần tử bằng nhau → 0 điểm gãy, cũng hợp lệ.

**Insight:** `nums` hợp lệ ⇔ **số điểm gãy trên vòng tròn ≤ 1**.

In [11]:
def drops_on_circle(nums):
    n = len(nums)
    return [i for i in range(n) if nums[i] > nums[(i + 1) % n]]

for a in [[1, 2, 3, 4, 5], [3, 4, 5, 1, 2], [2, 1, 3, 4], [1, 1, 1], [2, 1, 2]]:
    d = drops_on_circle(a)
    print(f"{str(a):<18} điểm gãy tại {d} -> số điểm gãy = {len(d)}")

[1, 2, 3, 4, 5]    điểm gãy tại [4] -> số điểm gãy = 1
[3, 4, 5, 1, 2]    điểm gãy tại [2] -> số điểm gãy = 1
[2, 1, 3, 4]       điểm gãy tại [0, 3] -> số điểm gãy = 2
[1, 1, 1]          điểm gãy tại [] -> số điểm gãy = 0
[2, 1, 2]          điểm gãy tại [0] -> số điểm gãy = 1


## 5. Lời giải chính — đếm điểm gãy trên vòng tròn

### 5.1. Một cặp kề nhau trên vòng tròn

Cặp thứ `i` là `(nums[i], nums[(i + 1) % n])`. Với `i = n - 1`, `(i + 1) % n = 0` → cặp (cuối, đầu).

In [12]:
nums = [3, 4, 5, 1, 2]
n = len(nums)
for i in range(n):
    j = (i + 1) % n
    print(f"i={i}, j={j}: ({nums[i]}, {nums[j]})", "<- GÃY" if nums[i] > nums[j] else "")

i=0, j=1: (3, 4) 
i=1, j=2: (4, 5) 
i=2, j=3: (5, 1) <- GÃY
i=3, j=4: (1, 2) 
i=4, j=0: (2, 3) 


### 5.2. Đếm và dừng sớm

Đếm số điểm gãy. Nếu vượt quá 1 thì trả `False` ngay — không cần duyệt tiếp.

### 5.3. Ghép lại

In [13]:
def check_drops(nums: list[int]) -> bool:
    n = len(nums)
    drops = 0
    for i in range(n):
        if nums[i] > nums[(i + 1) % n]:
            drops += 1
            if drops > 1:
                return False
    return True

for a in [[3, 4, 5, 1, 2], [2, 1, 3, 4], [1, 2, 3], [1, 1, 1], [2, 1, 2]]:
    print(a, "->", check_drops(a))

[3, 4, 5, 1, 2] -> True
[2, 1, 3, 4] -> False
[1, 2, 3] -> True
[1, 1, 1] -> True
[2, 1, 2] -> True


## 6. TRACE từng bước

Trace cho `nums = [3, 4, 5, 1, 2]` (n = 5):

| i | j=(i+1)%n | nums[i] | nums[j] | gãy? | drops |
|---:|---:|---:|---:|---|---:|
| 0 | 1 | 3 | 4 | Không | 0 |
| 1 | 2 | 4 | 5 | Không | 0 |
| 2 | 3 | 5 | 1 | **Có** | 1 |
| 3 | 4 | 1 | 2 | Không | 1 |
| 4 | 0 | 2 | 3 | Không | 1 |

Kết thúc với `drops = 1 ≤ 1` → `True`.

In [14]:
def trace_check(nums: list[int]) -> bool:
    n = len(nums)
    drops = 0
    print(f"nums = {nums}")
    for i in range(n):
        j = (i + 1) % n
        is_drop = nums[i] > nums[j]
        if is_drop:
            drops += 1
        print(f"  i={i}, j={j}: {nums[i]} > {nums[j]}? {is_drop!s:<5} | drops = {drops}")
        if drops > 1:
            print("  => drops > 1, trả False ngay")
            return False
    print(f"  => drops = {drops} <= 1, trả True")
    return True

trace_check([3, 4, 5, 1, 2])

nums = [3, 4, 5, 1, 2]
  i=0, j=1: 3 > 4? False | drops = 0
  i=1, j=2: 4 > 5? False | drops = 0
  i=2, j=3: 5 > 1? True  | drops = 1
  i=3, j=4: 1 > 2? False | drops = 1
  i=4, j=0: 2 > 3? False | drops = 1
  => drops = 1 <= 1, trả True


True

### 6.1. Trace ví dụ sai — điểm gãy thứ hai nằm ở **cặp vòng**

Với `[2, 1, 3, 4]`, nếu chỉ xét các cặp liền nhau thông thường ta chỉ thấy 1 điểm gãy (2 > 1). Điểm gãy thứ hai là cặp (cuối=4, đầu=2) — đây là lý do bắt buộc phải xét vòng.

In [15]:
trace_check([2, 1, 3, 4])
print()
trace_check([1, 1, 1])

nums = [2, 1, 3, 4]
  i=0, j=1: 2 > 1? True  | drops = 1
  i=1, j=2: 1 > 3? False | drops = 1
  i=2, j=3: 3 > 4? False | drops = 1
  i=3, j=0: 4 > 2? True  | drops = 2
  => drops > 1, trả False ngay

nums = [1, 1, 1]
  i=0, j=1: 1 > 1? False | drops = 0
  i=1, j=2: 1 > 1? False | drops = 0
  i=2, j=0: 1 > 1? False | drops = 0
  => drops = 0 <= 1, trả True


True

## 7. Code nộp LeetCode

In [16]:
class Solution:
    def check(self, nums: list[int]) -> bool:
        n = len(nums)
        drops = 0
        for i in range(n):
            if nums[i] > nums[(i + 1) % n]:
                drops += 1
                if drops > 1:
                    return False
        return True

sol = Solution()
print(sol.check([3, 4, 5, 1, 2]), sol.check([2, 1, 3, 4]), sol.check([1, 2, 3]))

True False True


## 8. Các lời giải khác

### 8.1. Một dòng với `sum` và `zip`

Ghép các cặp vòng bằng `zip(nums, nums[1:] + nums[:1])` (danh sách thứ hai là `nums` xoay trái 1). Trong Python, `True` được cộng như `1`.

In [17]:
print(True + True + False)
nums = [3, 4, 5, 1, 2]
print(list(zip(nums, nums[1:] + nums[:1])))

class SolutionOneLine:
    def check(self, nums: list[int]) -> bool:
        return sum(a > b for a, b in zip(nums, nums[1:] + nums[:1])) <= 1

print(SolutionOneLine().check([3, 4, 5, 1, 2]), SolutionOneLine().check([2, 1, 3, 4]))

2
[(3, 4), (4, 5), (5, 1), (1, 2), (2, 3)]
True False


### 8.2. Tìm điểm gãy rồi kiểm tra hai đoạn

Tìm điểm gãy **đầu tiên** `k` (chỗ `nums[k] > nums[k+1]`, không tính cặp vòng). Khi đó:

- Đoạn `nums[k+1:]` phải không giảm.
- Và phần tử cuối `nums[-1]` phải ≤ phần tử đầu `nums[0]` (để nối đuôi về đầu hợp lệ).

Nếu không có điểm gãy nào → mảng đã không giảm → `True`.

In [18]:
def check_split(nums: list[int]) -> bool:
    n = len(nums)
    k = -1
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            k = i
            break
    if k == -1:
        return True                        # đã sắp xếp sẵn (xoay 0)
    return is_non_decreasing(nums[k + 1:]) and nums[-1] <= nums[0]

for a in [[3, 4, 5, 1, 2], [2, 1, 3, 4], [1, 2, 3], [2, 1, 2], [3, 1, 2, 1]]:
    print(a, "->", check_split(a))

[3, 4, 5, 1, 2] -> True
[2, 1, 3, 4] -> False
[1, 2, 3] -> True
[2, 1, 2] -> True
[3, 1, 2, 1] -> False


In [19]:
class SolutionSplit:
    def check(self, nums: list[int]) -> bool:
        n = len(nums)
        for k in range(n - 1):
            if nums[k] > nums[k + 1]:
                rest = nums[k + 1:]
                return all(rest[i] <= rest[i + 1] for i in range(len(rest) - 1)) and nums[-1] <= nums[0]
        return True

print(SolutionSplit().check([3, 4, 5, 1, 2]), SolutionSplit().check([2, 1, 3, 4]))

True False


### 8.3. Bản nộp cho baseline thử mọi phép xoay

In [20]:
class SolutionBrute:
    def check(self, nums: list[int]) -> bool:
        n = len(nums)
        for x in range(n):
            r = nums[x:] + nums[:x]
            if all(r[i] <= r[i + 1] for i in range(n - 1)):
                return True
        return False

print(SolutionBrute().check([3, 4, 5, 1, 2]), SolutionBrute().check([2, 1, 3, 4]))

True False


### 8.4. Bảng so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---:|---:|---|
| Thử mọi phép xoay | xoay rồi kiểm tra không giảm | `O(n²)` | `O(n)` | Bám sát định nghĩa |
| So với xoay của `sorted` | mảng gốc phải là `sorted(nums)` | `O(n²)` | `O(n)` | Cũng bám định nghĩa |
| Đếm điểm gãy vòng | `drops <= 1` | `O(n)` | `O(1)` | **Lời giải chính** |
| Một dòng `sum/zip` | như trên | `O(n)` | `O(n)` (tạo list xoay) | Ngắn |
| Tìm gãy + kiểm hai đoạn | gãy đầu tiên, đoạn sau, nối đuôi | `O(n)` | `O(n)` (slice) | Dễ mở rộng để tìm số bước xoay |

## 9. Vì sao lời giải đúng?

**(⇒) Nếu `nums` là mảng xoay của mảng không giảm `A` thì số điểm gãy vòng ≤ 1.**
Trên vòng tròn, các cặp kề nhau của `nums` chính là các cặp kề nhau của `A` (xoay chỉ đổi điểm bắt đầu). `A` không giảm nên chỉ có thể gãy ở cặp `(A[n-1], A[0])`. Vậy ≤ 1 điểm gãy.

**(⇐) Nếu số điểm gãy vòng ≤ 1 thì `nums` hợp lệ.**
- 0 điểm gãy: đi vòng mà không bao giờ giảm thì `nums[0] ≤ nums[1] ≤ … ≤ nums[n-1] ≤ nums[0]` → mọi phần tử bằng nhau → hợp lệ.
- 1 điểm gãy tại `k`: bắt đầu đọc từ `k + 1` và đi vòng một lượt, ta không bao giờ gặp cặp giảm (điểm gãy duy nhất là cặp cuối cùng của lượt đọc, không được tính) → `nums[k+1:] + nums[:k+1]` không giảm → `nums` là phép xoay của mảng không giảm đó.

Ta kiểm chứng bằng máy: so với baseline trên **mọi** mảng nhỏ.

In [21]:
from itertools import product

checked = 0
for n in range(1, 7):
    for arr in product(range(1, 4), repeat=n):     # mọi mảng độ dài n, giá trị 1..3
        arr = list(arr)
        assert check_drops(arr) == check_bruteforce(arr) == check_compare_sorted(arr), arr
        checked += 1
print("Đối chiếu đầy đủ trên", checked, "mảng: đếm điểm gãy == thử mọi phép xoay")

Đối chiếu đầy đủ trên 1092 mảng: đếm điểm gãy == thử mọi phép xoay


## 10. Độ phức tạp

Gọi `n = len(nums)`.

- **Đếm điểm gãy:** một vòng `n` bước, mỗi bước `O(1)` → **Time `O(n)`**, **Space `O(1)`**.
- **Thử mọi phép xoay:** `n` phép xoay × (tạo mảng `O(n)` + kiểm tra `O(n)`) → **Time `O(n²)`**, **Space `O(n)`**.
- Không thể dưới `O(n)`: một phần tử bất kỳ chưa đọc có thể phá vỡ thứ tự.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Một phần tử; hai phần tử (tăng, giảm, bằng).
3. Có phần tử trùng: `[1,1,1]`, `[2,1,2]`, `[2,2,1,2]`, `[1,2,1,2]`.
4. Phản ví dụ cần cặp vòng: `[2,1,3,4]`, `[1,3,2]`.
5. Đối chiếu ngẫu nhiên các lớp `Solution`.

In [22]:
def check_case(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

sol = Solution()
check_case("đề 1", sol.check([3, 4, 5, 1, 2]), True)
check_case("đề 2", sol.check([2, 1, 3, 4]), False)
check_case("đề 3", sol.check([1, 2, 3]), True)
check_case("một phần tử", sol.check([5]), True)
check_case("hai phần tử tăng", sol.check([1, 2]), True)
check_case("hai phần tử giảm", sol.check([2, 1]), True)
check_case("hai phần tử bằng", sol.check([3, 3]), True)
check_case("toàn bằng nhau", sol.check([1, 1, 1]), True)
check_case("trùng [2,1,2]", sol.check([2, 1, 2]), True)
check_case("trùng [2,2,1,2]", sol.check([2, 2, 1, 2]), True)
check_case("trùng [1,2,1,2]", sol.check([1, 2, 1, 2]), False)
check_case("cần cặp vòng [1,3,2]", sol.check([1, 3, 2]), False)
check_case("giảm hẳn [3,2,1]", sol.check([3, 2, 1]), False)

đề 1                           PASS (True)
đề 2                           PASS (False)
đề 3                           PASS (True)
một phần tử                    PASS (True)
hai phần tử tăng               PASS (True)
hai phần tử giảm               PASS (True)
hai phần tử bằng               PASS (True)
toàn bằng nhau                 PASS (True)
trùng [2,1,2]                  PASS (True)
trùng [2,2,1,2]                PASS (True)
trùng [1,2,1,2]                PASS (False)
cần cặp vòng [1,3,2]           PASS (False)
giảm hẳn [3,2,1]               PASS (False)


In [23]:
import random

rng = random.Random(1752)
impls = [Solution(), SolutionOneLine(), SolutionSplit(), SolutionBrute()]
for _ in range(1000):
    n = rng.randint(1, 12)
    if rng.random() < 0.5:                          # sinh mảng hợp lệ: sắp xếp rồi xoay
        base = sorted(rng.randint(1, 100) for _ in range(n))
        x = rng.randrange(n)
        arr = base[x:] + base[:x]
    else:                                           # mảng ngẫu nhiên
        arr = [rng.randint(1, 5) for _ in range(n)]
    outs = {s.check(arr) for s in impls}
    assert len(outs) == 1, (arr, outs)
print("1000 mảng ngẫu nhiên: 4 lớp Solution khớp nhau")

1000 mảng ngẫu nhiên: 4 lớp Solution khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên cặp vòng (cuối, đầu)

Chỉ đếm điểm gãy giữa các phần tử liền kề thông thường.

In [24]:
def wrong_no_wrap(nums):
    drops = sum(nums[i] > nums[i + 1] for i in range(len(nums) - 1))
    return drops <= 1

print("sai:", wrong_no_wrap([2, 1, 3, 4]), "| đúng:", check_drops([2, 1, 3, 4]))

sai: True | đúng: False


**Sửa:** dùng `nums[(i + 1) % n]` và cho `i` chạy đủ `0..n-1`.

### Lỗi 2: Dùng `>=` thay vì `>` (coi phần tử bằng nhau là gãy)

Đề cho phép trùng, "không giảm" chấp nhận bằng nhau.

In [25]:
def wrong_ge(nums):
    n = len(nums)
    return sum(nums[i] >= nums[(i + 1) % n] for i in range(n)) <= 1

print("sai:", wrong_ge([1, 1, 1]), "| đúng:", check_drops([1, 1, 1]))

sai: False | đúng: True


### Lỗi 3: Chỉ kiểm "có đúng 1 điểm gãy" (`== 1`)

Mảng toàn phần tử bằng nhau có **0** điểm gãy trên vòng và vẫn hợp lệ.

In [26]:
def wrong_exactly_one(nums):
    n = len(nums)
    return sum(nums[i] > nums[(i + 1) % n] for i in range(n)) == 1

print("sai:", wrong_exactly_one([7]), wrong_exactly_one([2, 2]), "| đúng:", check_drops([7]), check_drops([2, 2]))

sai: False False | đúng: True True


**Sửa:** `<= 1`.

### Lỗi 4: Vượt biên khi đọc `nums[i + 1]` với `i = n - 1`

`range(n)` + `nums[i + 1]` → `IndexError` ở bước cuối. **Sửa:** `% n`, hoặc `range(n - 1)` rồi xét riêng cặp vòng.

In [27]:
nums = [3, 4, 5]
try:
    for i in range(len(nums)):
        _ = nums[i] > nums[i + 1]
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 5: Nghĩ rằng "xoay" nghĩa là chỉ cần `sorted(nums) == nums`

Như vậy chỉ chấp nhận xoay 0 vị trí → sai với `[3,4,5,1,2]`.

## 13. Mẹo nhận dạng pattern

- Đề có chữ **"rotated / circular / vòng"** → nghĩ tới **chỉ số `% n`** và đặc biệt **cặp (cuối, đầu)**.
- Mảng xoay của mảng sắp xếp có **≤ 1 điểm gãy** trên vòng — dấu hiệu dùng lại ở nhiều bài: tìm min trong mảng xoay (153), tìm kiếm trong mảng xoay (33).
- Kiểm tra "có phải ... sau một phép biến đổi" → baseline **thử mọi phép biến đổi**, rồi tìm **bất biến** mà phép biến đổi bảo toàn (ở đây: tập các cặp kề nhau trên vòng).
- Phần tử trùng → cẩn thận giữa `>` và `>=`.

Bài cùng họ: 153 Find Minimum in Rotated Sorted Array, 33 Search in Rotated Sorted Array, 189 Rotate Array, 796 Rotate String.

## 14. Biến thể

### 14.1. Tìm số bước xoay `x` (nếu hợp lệ)

Mảng gốc bắt đầu ngay **sau** điểm gãy. Nếu không có điểm gãy thì `x = 0`. Trả `-1` nếu không hợp lệ. (Với mảng toàn bằng nhau, chọn `x = 0`.)

In [28]:
def rotation_offset(nums):
    n = len(nums)
    drops = [i for i in range(n) if nums[i] > nums[(i + 1) % n]]
    if len(drops) > 1:
        return -1
    if not drops:
        return 0
    return (drops[0] + 1) % n      # chỉ số của phần tử đầu tiên của mảng gốc trong nums

print(rotation_offset([3, 4, 5, 1, 2]))   # 3 -> nums[3:] + nums[:3] = [1,2,3,4,5]
print(rotation_offset([1, 2, 3]))         # 0
print(rotation_offset([2, 1, 3, 4]))      # -1

3
0
-1


### 14.2. Mảng xoay của mảng **không tăng**

Đối xứng: đổi `>` thành `<`.

In [29]:
def check_non_increasing_rotated(nums):
    n = len(nums)
    return sum(nums[i] < nums[(i + 1) % n] for i in range(n)) <= 1

print(check_non_increasing_rotated([2, 1, 5, 4, 3]))  # xoay [5,4,3,2,1]
print(check_non_increasing_rotated([1, 2, 3]))         # False

True
False


`[1, 2, 3]` trên vòng: `1<2` (1), `2<3` (2), `3<1`? không → 2 "điểm tăng" → `False`. Đúng: `[1,2,3]` không phải xoay của dãy không tăng nào.

### 14.3. Bài 796 — Rotate String

Chuỗi `goal` là phép xoay của `s` ⇔ cùng độ dài và `goal` nằm trong `s + s`. Ý tưởng "nhân đôi để mô phỏng vòng tròn" rất đáng nhớ.

In [30]:
def rotate_string(s, goal):
    return len(s) == len(goal) and goal in s + s

print(rotate_string("abcde", "cdeab"), rotate_string("abcde", "abced"))

True False


## 15. Bài tập tự luyện

Các hàm TODO không được gọi khi chạy notebook. Đáp án ở mục 17.

### Bài 1 — Tính tay

Đếm số điểm gãy trên vòng cho `[6, 7, 1, 2, 3]`, `[1, 3, 2, 4]`, `[5, 5, 5, 1]`. Mảng nào hợp lệ?

### Bài 2 — Dự đoán output

Với `a = [10, 20, 30, 40]`: `a[1:] + a[:1]`, `(3 + 1) % 4`, `sum(x > 15 for x in a)` lần lượt ra gì?

### Bài 3 — Điền TODO: đếm điểm gãy

In [31]:
def exercise_check(nums: list[int]) -> bool:
    # TODO 1: n = len(nums), drops = 0
    # TODO 2: với i trong range(n): so nums[i] với nums[(i + 1) % n]
    # TODO 3: trả drops <= 1
    pass

# assert exercise_check([3, 4, 5, 1, 2]) == True
# assert exercise_check([2, 1, 3, 4]) == False

### Bài 4 — Tự viết baseline thử mọi phép xoay

In [32]:
def exercise_check_brute(nums: list[int]) -> bool:
    # TODO: với mỗi x, r = nums[x:] + nums[:x]; nếu r không giảm thì True
    pass

# assert exercise_check_brute([1, 1, 1]) == True
# assert exercise_check_brute([1, 3, 2]) == False

### Bài 5 — Biến thể: phần tử nhỏ nhất của mảng xoay hợp lệ, một lần duyệt

Cho `nums` **đảm bảo** là mảng xoay của mảng không giảm. Trả phần tử nhỏ nhất **không dùng `min()`**, mà dựa vào điểm gãy.

In [33]:
def exercise_min_rotated(nums: list[int]) -> int:
    # TODO: phần tử ngay sau điểm gãy là nhỏ nhất; không có điểm gãy -> nums[0]
    pass

# assert exercise_min_rotated([3, 4, 5, 1, 2]) == 1
# assert exercise_min_rotated([1, 2, 3]) == 1

### Bài 6 — Mini challenge: xóa tối đa một phần tử

Trả `True` nếu có thể **xóa tối đa một phần tử** để mảng còn lại là mảng xoay của mảng không giảm. (n nhỏ, được phép thử từng vị trí xóa.)

In [34]:
def exercise_check_after_one_removal(nums: list[int]) -> bool:
    # TODO: thử không xóa, rồi thử xóa từng i (nums[:i] + nums[i+1:])
    pass

# assert exercise_check_after_one_removal([2, 1, 3, 4]) == True     # xóa 2 -> [1,3,4]
# assert exercise_check_after_one_removal([3, 1, 2, 1, 2]) == True
# assert exercise_check_after_one_removal([1, 3, 2, 4, 3]) == False

## 16. Gợi ý 3 mức

**Bài 1** — M1: nhớ xét cặp (cuối, đầu). M2: `[6,7,1,2,3]`: chỉ `7>1`. M3: số điểm gãy lần lượt 1, 2, 1.

**Bài 2** — M1: slicing + nối. M2: `%` đưa 4 về 0. M3: `[20, 30, 40, 10]`, `0`, `3`.

**Bài 3** — M1: một vòng `range(n)`. M2: `j = (i + 1) % n`. M3: `if nums[i] > nums[j]: drops += 1`.

**Bài 4** — M1: có `n` cách xoay. M2: kiểm tra không giảm bằng `all(r[i] <= r[i+1] ...)`. M3: trả `False` nếu không `x` nào thỏa.

**Bài 5** — M1: phần tử nhỏ nhất là "đầu" của mảng gốc. M2: đầu mảng gốc nằm ngay sau điểm gãy. M3: tìm `i` với `nums[i] > nums[i+1]`, trả `nums[i+1]`; không có thì `nums[0]`.

**Bài 6** — M1: tái sử dụng hàm kiểm tra chính. M2: `nums[:i] + nums[i+1:]` là mảng đã xóa phần tử `i`. M3: `check(nums) or any(check(nums[:i] + nums[i+1:]) for i in range(n))`.

## 17. Đáp án tham khảo

**Bài 1:** `[6,7,1,2,3]` → 1 điểm gãy (`7>1`), hợp lệ. `[1,3,2,4]` → 2 điểm gãy (`3>2`, `4>1`), không hợp lệ. `[5,5,5,1]` → 1 điểm gãy (`5>1`), hợp lệ.

**Bài 2:** `[20, 30, 40, 10]`, `0`, `3`.

In [35]:
for a in [[6, 7, 1, 2, 3], [1, 3, 2, 4], [5, 5, 5, 1]]:
    print(a, "gãy tại", drops_on_circle(a), "->", check_drops(a))
a = [10, 20, 30, 40]
print(a[1:] + a[:1], (3 + 1) % 4, sum(x > 15 for x in a))

[6, 7, 1, 2, 3] gãy tại [1] -> True
[1, 3, 2, 4] gãy tại [1, 3] -> False
[5, 5, 5, 1] gãy tại [2] -> True
[20, 30, 40, 10] 0 3


In [36]:
def answer_check(nums):
    n = len(nums)
    drops = 0
    for i in range(n):
        if nums[i] > nums[(i + 1) % n]:
            drops += 1
    return drops <= 1


def answer_check_brute(nums):
    n = len(nums)
    for x in range(n):
        r = nums[x:] + nums[:x]
        if all(r[i] <= r[i + 1] for i in range(n - 1)):
            return True
    return False


def answer_min_rotated(nums):
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            return nums[i + 1]
    return nums[0]


def answer_check_after_one_removal(nums):
    n = len(nums)
    if answer_check(nums):
        return True
    return any(answer_check(nums[:i] + nums[i + 1:]) for i in range(n))


assert answer_check([3, 4, 5, 1, 2]) == True
assert answer_check([2, 1, 3, 4]) == False
assert answer_check_brute([1, 1, 1]) == True
assert answer_check_brute([1, 3, 2]) == False
assert answer_min_rotated([3, 4, 5, 1, 2]) == 1
assert answer_min_rotated([1, 2, 3]) == 1
assert answer_check_after_one_removal([2, 1, 3, 4]) == True
assert answer_check_after_one_removal([3, 1, 2, 1, 2]) == True
assert answer_check_after_one_removal([1, 3, 2, 4, 3]) == False
rng = random.Random(52)
for _ in range(300):
    base = sorted(rng.randint(1, 9) for _ in range(rng.randint(1, 10)))
    x = rng.randrange(len(base))
    arr = base[x:] + base[:x]
    assert answer_min_rotated(arr) == min(arr)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


Giải thích test `[3, 1, 2, 1, 2]` của bài 6: xóa phần tử `1` ở chỉ số 3 → `[3, 1, 2, 2]`, trên vòng chỉ gãy ở `3 > 1` → hợp lệ (xoay của `[1, 2, 2, 3]`). Còn `[1, 3, 2, 4, 3]` (3 điểm gãy trên vòng: `3>2`, `4>3`, `3>1`) thử mọi cách xóa đều còn ≥ 2 điểm gãy.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def check(self, nums: list[int]) -> bool:
        n = len(nums)
        drops = 0
        for i in range(n):
            if nums[i] > nums[(i + 1) % n]:   # xét cả cặp (cuối, đầu)
                drops += 1
        return drops <= 1                     # 0 hoặc 1 điểm gãy
```

Template duyệt vòng:

```text
for i in range(n):
    j = (i + 1) % n        # phần tử kế tiếp trên vòng tròn
    xử_lý(nums[i], nums[j])
```

Checklist trước khi submit:

- [ ] Có xét cặp vòng `(nums[n-1], nums[0])`?
- [ ] Dùng `>` (không phải `>=`) vì cho phép phần tử bằng nhau?
- [ ] Điều kiện là `<= 1` (mảng toàn bằng nhau có 0 điểm gãy)?
- [ ] Không vượt biên khi đọc phần tử kế tiếp?
- [ ] Đã thử `[2,1,3,4]`, `[1,1,1]`, `[2,1,2]`, `[5]`?